# 05.03 CatBoost Classification (Patient-Grouped 10-Fold CV, `allow_writing_files=False`)

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, brier_score_loss, log_loss, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import CalibrationDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

RANDOM_SEED = 42

def find_project_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur] + list(cur.parents):
        if (p / "data").exists() and (p / "experiments").exists():
            return p
    return cur

PROJECT_ROOT = find_project_root()
MODELS_DIR = PROJECT_ROOT / "models" / "phase_05"
OUT_CB_DIR = PROJECT_ROOT / "models" / "trained" / "gradient_boosting"
VIZ_DIR = PROJECT_ROOT / "reports" / "phase_05" / "visualizations" / "catboost"
OUT_CB_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

split = json.loads((MODELS_DIR / "phase05_split_and_schema.json").read_text(encoding="utf-8"))
df_auth = pd.read_csv(PROJECT_ROOT / split["dataset_authentic"])
df_syn = pd.read_csv(PROJECT_ROOT / split["dataset_synthetic"])

NUM_COLS = split["num_cols"]
CAT_COLS = split["cat_cols"]
FEATURE_COLS = split["feature_cols"]

df_tr_dev = df_auth.iloc[split["auth_train_indices"]].reset_index(drop=True)
X_dev = df_tr_dev[FEATURE_COLS].copy()
y_dev = df_tr_dev["target"].to_numpy(dtype=int)
folds = split["cv_folds"]

def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_t = np.asarray(y_true, dtype=int)
    y_p = np.clip(np.asarray(y_prob, dtype=float), 1e-7, 1.0 - 1e-7)
    y_pred = (y_p >= threshold).astype(int)
    cm = confusion_matrix(y_t, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    bal_acc = 0.5 * (spec + rec)
    bins = np.linspace(0.0, 1.0, 11)
    ece = 0.0
    for i in range(10):
        m = (y_p >= bins[i]) & (y_p < bins[i + 1] if i < 9 else y_p <= bins[i + 1])
        if m.any():
            ece += (m.sum() / len(y_t)) * abs(y_t[m].mean() - y_p[m].mean())
    auc = float(roc_auc_score(y_t, y_p)) if len(np.unique(y_t)) == 2 else float("nan")
    return {
        "accuracy": float(accuracy_score(y_t, y_pred)),
        "precision": float(precision_score(y_t, y_pred, pos_label=1, zero_division=0)),
        "recall": float(rec),
        "specificity": float(spec),
        "balanced_accuracy": float(bal_acc),
        "f1": float(f1_score(y_t, y_pred, pos_label=1, zero_division=0)),
        "macro_f1": float(f1_score(y_t, y_pred, average="macro", zero_division=0)),
        "roc_auc": auc,
        "brier_score": float(brier_score_loss(y_t, y_p)),
        "log_loss": float(log_loss(y_t, y_p, labels=[0, 1])),
        "ece": float(ece),
        "confusion_matrix": cm.tolist(),
    }

def build_cb_pipe():
    prep = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM_COLS),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="UNRECORDED")), ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]), CAT_COLS),
    ])
    clf = CatBoostClassifier(
        iterations=250, depth=4, learning_rate=0.04, l2_leaf_reg=5.0,
        auto_class_weights="Balanced", random_seed=RANDOM_SEED,
        thread_count=2, allow_writing_files=False, verbose=0,
    )
    return Pipeline([("prep", prep), ("clf", clf)])

def run_cv(augment_synthetic=False):
    oof_probs = np.zeros(len(y_dev), dtype=float)
    fold_rows = []
    for f in folds:
        tr_i, va_i = f["train_indices"], f["val_indices"]
        X_tr, y_tr = X_dev.iloc[tr_i].copy(), y_dev[tr_i].copy()
        if augment_synthetic:
            tr_pts = set(df_tr_dev.iloc[tr_i]["patient_group_id"])
            va_pts = set(df_tr_dev.iloc[va_i]["patient_group_id"])
            syn_f = df_syn[df_syn["patient_group_id"].isin(tr_pts)]
            assert len(set(syn_f["patient_group_id"]) & va_pts) == 0
            if len(syn_f) > 0:
                X_tr = pd.concat([X_tr, syn_f[FEATURE_COLS]], ignore_index=True)
                y_tr = np.concatenate([y_tr, syn_f["target"].to_numpy(dtype=int)])
        pipe = build_cb_pipe()
        pipe.fit(X_tr, y_tr)
        probs = pipe.predict_proba(X_dev.iloc[va_i])[:, 1]
        oof_probs[va_i] = probs
        m = compute_binary_metrics(y_dev[va_i], probs)
        m["fold"] = f["fold"]
        fold_rows.append(m)
    overall = compute_binary_metrics(y_dev, oof_probs)
    return oof_probs, pd.DataFrame(fold_rows), overall

oof_a, folds_a, overall_a = run_cv(augment_synthetic=False)
oof_b, folds_b, overall_b = run_cv(augment_synthetic=True)

print("CatBoost Track A (Authentic-Only) 10-Fold OOF:", json.dumps(overall_a, indent=2))
print("CatBoost Track B (Synthetic-Augmented Train, Authentic Val) 10-Fold OOF:", json.dumps(overall_b, indent=2))

cb_final_a = build_cb_pipe().fit(X_dev, y_dev)
joblib.dump(cb_final_a, OUT_CB_DIR / "catboost_model.joblib")

syn_tr = df_syn.iloc[split["syn_train_indices"]]
cb_final_b = build_cb_pipe().fit(
    pd.concat([X_dev, syn_tr[FEATURE_COLS]], ignore_index=True),
    np.concatenate([y_dev, syn_tr["target"].to_numpy(dtype=int)]),
)
joblib.dump(cb_final_b, OUT_CB_DIR / "catboost_model_trackB.joblib")

np.save(MODELS_DIR / "cb_oof_probs.npy", oof_a)
np.save(MODELS_DIR / "cb_trackB_oof_probs.npy", oof_b)

# Correctly aligned feature names via get_feature_names_out()
feat_names = list(cb_final_a.named_steps["prep"].get_feature_names_out())
imps = cb_final_a.named_steps["clf"].get_feature_importance()
df_imp = pd.DataFrame({"feature": feat_names, "importance": imps}).sort_values("importance", ascending=False)
df_imp.to_csv(VIZ_DIR / "feature_importances.csv", index=False)

payload = {
    "model_name": "CatBoost",
    "cv_protocol": "10-Fold StratifiedGroupKFold (Authentic Patients)",
    "n_train_dev_patients": int(len(y_dev)),
    "track_a_oof": overall_a,
    "track_a_fold_mean": folds_a.drop(columns=["fold", "confusion_matrix"]).mean().to_dict(),
    "track_a_fold_std": folds_a.drop(columns=["fold", "confusion_matrix"]).std().to_dict(),
    "track_b_oof": overall_b,
    "top_features": df_imp.head(15).to_dict(orient="records"),
}
(OUT_CB_DIR / "catboost_metrics.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
folds_a.drop(columns=["confusion_matrix"]).to_csv(VIZ_DIR / "evaluation_summary_table.csv", index=False)

# Plots
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_dev, (oof_a >= 0.5).astype(int), display_labels=["INELIGIBLE", "ELIGIBLE"], ax=axes[0], cmap="Blues")
axes[0].set_title("CatBoost — Track A OOF Confusion Matrix")
RocCurveDisplay.from_predictions(y_dev, oof_a, ax=axes[1]); axes[1].set_title("CatBoost — Track A ROC Curve")
plt.tight_layout(); plt.savefig(VIZ_DIR / "confusion_matrix_and_roc.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
top12 = df_imp.head(12).iloc[::-1]
ax.barh(top12["feature"], top12["importance"], color="#41b6c4", edgecolor="black")
ax.set_title("CatBoost — Top 12 Aligned Feature Importances (Track A)")
plt.tight_layout(); plt.savefig(VIZ_DIR / "feature_importance.png", dpi=200); plt.close(fig)
print("Saved CatBoost artifacts and plots.")


CatBoost Track A (Authentic-Only) 10-Fold OOF: {
  "accuracy": 0.9473684210526315,
  "precision": 0.9365079365079365,
  "recall": 0.9833333333333333,
  "specificity": 0.8857142857142857,
  "balanced_accuracy": 0.9345238095238095,
  "f1": 0.959349593495935,
  "macro_f1": 0.9423613639121466,
  "roc_auc": 0.9600000000000001,
  "brier_score": 0.05143120095552172,
  "log_loss": 0.19221184361268034,
  "ece": 0.06815612534859408,
  "confusion_matrix": [
    [
      31,
      4
    ],
    [
      1,
      59
    ]
  ]
}
CatBoost Track B (Synthetic-Augmented Train, Authentic Val) 10-Fold OOF: {
  "accuracy": 0.9368421052631579,
  "precision": 0.9354838709677419,
  "recall": 0.9666666666666667,
  "specificity": 0.8857142857142857,
  "balanced_accuracy": 0.9261904761904762,
  "f1": 0.9508196721311475,
  "macro_f1": 0.9312921890067503,
  "roc_auc": 0.9323809523809523,
  "brier_score": 0.05264285214714483,
  "log_loss": 0.27526757454777817,
  "ece": 0.05037524612116007,
  "confusion_matrix": [
    

Saved CatBoost artifacts and plots.
